In [0]:
# ============================================================
# PROJETO: UCI Online Retail
# CAMADA: SILVER
# ============================================================
#
# Objetivo:
# Limpar, padronizar e enriquecer os dados provenientes
# da camada Bronze.
#
# Princípios:
# 1. Preservar a rastreabilidade dos dados.
# 2. Não alterar a tabela Bronze original.
# 3. Documentar todas as transformações.
# 4. Registrar regras de negócio nos metadados.
# 5. Preparar os dados para análises e agregações na Gold.
# 6. Fornecer contexto estruturado para consumo futuro
#    por ferramentas analíticas e soluções de IA.
#
# Fonte:
# UCI Online Retail Dataset
#
# Camada de origem:
# Bronze — dados próximos à fonte original.
#
# Camada de destino:
# Silver — dados limpos, padronizados e enriquecidos.
# ============================================================

In [0]:
# ============================================================
# METADADOS — Carga da camada Bronze
# ============================================================
# Origem:
# Tabela Delta localizada na camada Bronze.
#
# Objetivo:
# Utilizar os dados originais armazenados na Bronze como
# fonte para as transformações da camada Silver.
#
# Regra:
# A tabela Bronze não será modificada diretamente.
# Todas as transformações serão realizadas em um novo
# DataFrame destinado à camada Silver.
# ============================================================

silver_df = spark.table("uci_retail.bronze.online_retail")

In [0]:
silver_df.count()

541909

In [0]:
display(silver_df.limit(10))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
554512,23280,FOLDING BUTTERFLY MIRROR HOT PINK,1,2011-05-24T15:54:00.000Z,2.46,null,United Kingdom
554512,23298,SPOTTY BUNTING,1,2011-05-24T15:54:00.000Z,10.79,null,United Kingdom
554512,23302,KNEELING MAT HOUSEWORK DESIGN,2,2011-05-24T15:54:00.000Z,3.29,null,United Kingdom
554512,35924,HANGING FAIRY CAKE DECORATION,1,2011-05-24T15:54:00.000Z,4.13,null,United Kingdom
554512,37476,CONDIMENT TRAY 4 BOWLS AND 4 SPOONS,1,2011-05-24T15:54:00.000Z,8.29,null,United Kingdom
554512,47566,PARTY BUNTING,1,2011-05-24T15:54:00.000Z,10.79,null,United Kingdom
554512,48194,DOORMAT HEARTS,1,2011-05-24T15:54:00.000Z,15.79,null,United Kingdom
554512,82567,"AIRLINE LOUNGE,METAL SIGN",1,2011-05-24T15:54:00.000Z,1.63,null,United Kingdom
554512,84380,SET OF 3 BUTTERFLY COOKIE CUTTERS,1,2011-05-24T15:54:00.000Z,2.46,null,United Kingdom
554512,84596B,SMALL DOLLY MIX DESIGN ORANGE BOWL,2,2011-05-24T15:54:00.000Z,0.83,null,United Kingdom


In [0]:
# ============================================================
# METADADOS — TransactionType
# ============================================================
# Nome: TransactionType
# Tipo: STRING
# Camada: Silver
# Origem: InvoiceNo
#
# Descrição:
# Classifica cada registro de acordo com o tipo de transação.
#
# Regra de negócio:
# - InvoiceNo iniciado por "C" = Cancellation
# - Demais registros = Sale
#
# Objetivo:
# Permitir a diferenciação entre vendas e cancelamentos
# sem excluir os registros de cancelamento.
#
# Rastreabilidade:
# InvoiceNo original permanece preservado.
#
# Observação:
# Foram identificados anteriormente 9.288 registros cujo
# InvoiceNo começa com "C".
# ============================================================

from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    "TransactionType",
    when(col("InvoiceNo").startswith("C"), "Cancellation")
    .otherwise("Sale")
)

In [0]:
display(
    silver_df.groupBy("TransactionType").count()
)

TransactionType,count
Sale,532621
Cancellation,9288


In [0]:
silver_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- TransactionType: string (nullable = false)



In [0]:
from pyspark.sql.functions import col, sum

display(
    silver_df.select(
        [
            sum(col(c).isNull().cast("int")).alias(c)
            for c in silver_df.columns
        ]
    )
)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionType
0,0,0,0,0,0,135080,0,0


In [0]:
# ============================================================
# METADADOS — CustomerID
# ============================================================
# Nome: CustomerID
# Tipo original: DOUBLE
# Tipo planejado na Silver: STRING
# Origem: Dataset UCI Online Retail
# Camada: Silver
#
# Descrição:
# Identificador único associado ao cliente responsável pela
# transação.
#
# Qualidade dos dados:
# Foram identificados 135.080 registros sem CustomerID.
#
# Tratamento:
# Os valores NULL serão preservados.
#
# Justificativa:
# A ausência de CustomerID não significa que a transação
# seja inválida. A venda pode possuir informações válidas
# de produto, quantidade, preço e data mesmo sem identificação
# do cliente.
#
# Regra para IA:
# NULL em CustomerID significa "cliente não identificado /
# informação de cliente não disponível".
# NULL NÃO deve ser interpretado como CustomerID = 0.
#
# Impacto analítico:
# Métricas baseadas em clientes identificados devem considerar
# somente registros que possuam CustomerID.
# Métricas gerais de vendas podem utilizar registros sem
# CustomerID, desde que a regra da métrica permita.
# ============================================================

In [0]:
display(
    silver_df
    .filter(
        col("CustomerID").isNotNull() &
        (col("CustomerID") != col("CustomerID").cast("long"))
    )
    .select("CustomerID")
    .limit(20)
)

CustomerID


In [0]:
# ============================================================
# METADADOS — Padronização do CustomerID
# ============================================================
# Campo: CustomerID
#
# Tipo original:
# DOUBLE
#
# Tipo Silver:
# STRING
#
# Motivo:
# CustomerID é um identificador de cliente e não uma medida
# numérica. Portanto, não deve ser utilizado em operações
# matemáticas.
#
# Validação:
# Foi verificado que os CustomerID não possuem valores
# decimais reais entre os registros não nulos.
#
# Tratamento de NULL:
# Os 135.080 valores NULL serão preservados.
#
# Regra para IA:
# CustomerID deve ser interpretado como identificador.
# Não realizar soma, média ou outras operações matemáticas
# sobre esse campo.
# ============================================================

In [0]:
silver_df = silver_df.withColumn(
    "CustomerID",
    col("CustomerID").cast("long").cast("string")
)

In [0]:
silver_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- TransactionType: string (nullable = false)



In [0]:
silver_df.filter(col("CustomerID").isNull()).count()

135080

In [0]:
display(
    silver_df
    .select("Quantity")
    .summary("count", "min", "max", "mean", "stddev")
)

summary,Quantity
count,541909
min,-80995
max,80995
mean,9.55224954743324
stddev,218.0811578502346


In [0]:
display(
    silver_df
    .filter(col("Quantity") < 0)
    .select("InvoiceNo", "StockCode", "Quantity", "UnitPrice", "TransactionType")
    .limit(20)
)

InvoiceNo,StockCode,Quantity,UnitPrice,TransactionType
C547899,POST,-1,55.89,Cancellation
C547899,M,-1,1486.12,Cancellation
C547904,AMAZONFEE,-1,219.76,Cancellation
C547905,POST,-1,18.66,Cancellation
C547905,M,-1,416.75,Cancellation
C547908,M,-1,646.46,Cancellation
C547908,POST,-1,38.45,Cancellation
C547913,POST,-1,13.39,Cancellation
C547913,M,-1,325.0,Cancellation
C547915,POST,-1,29.7,Cancellation


In [0]:
# ============================================================
# METADADOS — Quantity
# ============================================================
# Campo: Quantity
# Tipo: LONG
# Origem: Dataset UCI Online Retail
# Camada: Silver
#
# Descrição:
# Quantidade de unidades registrada na linha da transação.
#
# Validação:
# Valores negativos serão investigados antes de qualquer
# remoção ou transformação.
#
# Regra de negócio:
# A interpretação de valores negativos deve considerar o
# TransactionType e o contexto da transação.
#
# Regra para IA:
# Quantity representa quantidade de unidades e não deve ser
# interpretada automaticamente como valor monetário.
# ============================================================

In [0]:
display(
    silver_df
    .select("Quantity")
    .summary("count", "min", "max", "mean", "stddev")
)

summary,Quantity
count,541909
min,-80995
max,80995
mean,9.55224954743324
stddev,218.0811578502346


In [0]:
display(
    silver_df
    .filter(col("Quantity") < 0)
    .select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "TransactionType"
    )
    .limit(20)
)

InvoiceNo,StockCode,Quantity,UnitPrice,TransactionType
C547899,POST,-1,55.89,Cancellation
C547899,M,-1,1486.12,Cancellation
C547904,AMAZONFEE,-1,219.76,Cancellation
C547905,POST,-1,18.66,Cancellation
C547905,M,-1,416.75,Cancellation
C547908,M,-1,646.46,Cancellation
C547908,POST,-1,38.45,Cancellation
C547913,POST,-1,13.39,Cancellation
C547913,M,-1,325.0,Cancellation
C547915,POST,-1,29.7,Cancellation


In [0]:
display(
    silver_df
    .select("UnitPrice")
    .summary("count", "min", "max", "mean", "stddev")
)

summary,UnitPrice
count,541909
min,-11062.06
max,38970.0
mean,4.611113626089597
stddev,96.75985306117931


In [0]:
display(
    silver_df
    .filter(col("UnitPrice") <= 0)
    .select(
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "TransactionType"
    )
    .limit(30)
)

InvoiceNo,StockCode,Description,Quantity,UnitPrice,TransactionType
547916,22459,nan,-36,0.0,Sale
547917,21898,nan,6,0.0,Sale
547927,37444C,nan,5,0.0,Sale
547928,37444B,nan,1,0.0,Sale
547929,21393,nan,3,0.0,Sale
547932,85197,nan,24,0.0,Sale
547933,21445,nan,1,0.0,Sale
547943,21357,nan,1,0.0,Sale
547948,71038,nan,167,0.0,Sale
547950,35241,damages?,-40,0.0,Sale


In [0]:
from pyspark.sql.functions import when, col, count, sum

display(
    silver_df
    .groupBy("TransactionType")
    .agg(
        count("*").alias("TotalRecords"),
        sum(when(col("UnitPrice") == 0, 1).otherwise(0)).alias("ZeroUnitPrice"),
        sum(when(col("UnitPrice") < 0, 1).otherwise(0)).alias("NegativeUnitPrice"),
        sum(when(col("Quantity") < 0, 1).otherwise(0)).alias("NegativeQuantity")
    )
)

TransactionType,TotalRecords,ZeroUnitPrice,NegativeUnitPrice,NegativeQuantity
Sale,532621,2515,2,1336
Cancellation,9288,0,0,9288


In [0]:
silver_df = silver_df.withColumn(
    "DataQualityStatus",
    when(
        (col("Quantity") <= 0) | (col("UnitPrice") <= 0),
        "Review"
    ).otherwise("Valid")
)

In [0]:
display(
    silver_df
    .groupBy("DataQualityStatus")
    .count()
)

DataQualityStatus,count
Review,11805
Valid,530104


In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

In [0]:
display(
    silver_df
    .select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "TransactionType",
        "DataQualityStatus",
        "Revenue"
    )
    .limit(20)
)

InvoiceNo,StockCode,Quantity,UnitPrice,TransactionType,DataQualityStatus,Revenue
554512,23280,1,2.46,Sale,Valid,2.46
554512,23298,1,10.79,Sale,Valid,10.79
554512,23302,2,3.29,Sale,Valid,6.58
554512,35924,1,4.13,Sale,Valid,4.13
554512,37476,1,8.29,Sale,Valid,8.29
554512,47566,1,10.79,Sale,Valid,10.79
554512,48194,1,15.79,Sale,Valid,15.79
554512,82567,1,1.63,Sale,Valid,1.63
554512,84380,1,2.46,Sale,Valid,2.46
554512,84596B,2,0.83,Sale,Valid,1.66


In [0]:
display(
    silver_df
    .select("Revenue")
    .summary("count", "min", "max", "mean", "stddev")
)

summary,Revenue
count,541909
min,-168469.6
max,168469.6
mean,17.987794876998628
stddev,378.81082350597546


In [0]:
display(
    silver_df
    .select(
        "TransactionType",
        "DataQualityStatus",
        "Revenue"
    )
    .groupBy(
        "TransactionType",
        "DataQualityStatus"
    )
    .agg(
        count("*").alias("TotalRecords"),
        sum("Revenue").alias("TotalRevenue")
    )
)

TransactionType,DataQualityStatus,TotalRecords,TotalRevenue
Cancellation,Review,9288,-896812.4899999977
Sale,Valid,530104,1.0666684543999156E7
Sale,Review,2517,-22124.12


In [0]:
silver_df = silver_df.withColumn(
    "DataQualityStatus",
    when(
        col("TransactionType") == "Cancellation",
        "Valid"
    ).when(
        (col("Quantity") <= 0) | (col("UnitPrice") <= 0),
        "Review"
    ).otherwise("Valid")
)

In [0]:
display(
    silver_df
    .groupBy("TransactionType", "DataQualityStatus")
    .count()
)

TransactionType,DataQualityStatus,count
Cancellation,Valid,9288
Sale,Valid,530104
Sale,Review,2517


In [0]:
# ============================================================
# METADADOS — Revenue
# ============================================================
# Nome: Revenue
# Tipo: DOUBLE
# Camada: Silver
# Origem: Quantity e UnitPrice
#
# Descrição:
# Representa o impacto financeiro de cada linha da transação.
#
# Regra:
# Revenue = Quantity * UnitPrice
#
# Interpretação:
# - Valor positivo: impacto financeiro de uma venda.
# - Valor negativo: impacto financeiro de um cancelamento.
# - Valor zero: linha sem impacto financeiro.
#
# Observação:
# Cancelamentos não são removidos da Silver. Quando a quantidade
# é negativa, o Revenue também representa um valor negativo,
# permitindo analisar posteriormente o impacto dos cancelamentos.
#
# Uso futuro:
# A coluna será utilizada na camada Gold para criação de métricas
# como receita, vendas por período, país e produto.
# ============================================================

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.silver.online_retail")

In [0]:
silver_check = spark.table("uci_retail.silver.online_retail")

print("Quantidade de registros:", silver_check.count())

silver_check.printSchema()

Quantidade de registros: 541909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- DataQualityStatus: string (nullable = true)
 |-- Revenue: double (nullable = true)



In [0]:
display(
    silver_check
    .select(
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "TransactionType",
        "DataQualityStatus",
        "Revenue"
    )
    .limit(20)
)

InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID,TransactionType,DataQualityStatus,Revenue
536365,85123A,6,2.55,17850,Sale,Valid,15.299999999999999
536365,71053,6,3.39,17850,Sale,Valid,20.34
536365,84406B,8,2.75,17850,Sale,Valid,22.0
536365,84029G,6,3.39,17850,Sale,Valid,20.34
536365,84029E,6,3.39,17850,Sale,Valid,20.34
536365,22752,2,7.65,17850,Sale,Valid,15.3
536365,21730,6,4.25,17850,Sale,Valid,25.5
536366,22633,6,1.85,17850,Sale,Valid,11.100000000000001
536366,22632,6,1.85,17850,Sale,Valid,11.100000000000001
536367,84879,32,1.69,13047,Sale,Valid,54.08
